# 02 – Construction des indicateurs et agrégations
## Projet : Répartition des ressources et charge de travail

Ce notebook sert à préparer les données pour la visualisation et le tableau de bord.
Il inclut :

- un nettoyage,
- la création des indicateurs métier,
- les agrégations par centre, phase, module,
- l'export final vers un fichier Excel.

Ce fichier sera utilisé dans Tableau Public.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel(r"C:\Users\Tenordem\Documents\projet-repartition-charge\data\raw\data_complet_v2.xlsx")
df.head()


,NOM_SI,ModuleId,PhaseId,Types_phases,utilisateur_id2,Centre,DateDebutPrev,DateDebutReelle,DateFinPrevisionnelle,DateFinReelle,DateFinActualisée,charge_previsionnelle_n,ChargeTotaleEstimee,ChargeTotaleActualisee,ModuleChargeInitiale,ChargeGarantie
0,SI_HORS-PROCESSUS,5477,10407,AUTRES,-,-,2024-01-01 00:00:00,2024-01-01 00:00:00,2050-12-31 00:00:00,2050-12-31 00:00:00,-,0,2300,2300,0,-
1,SI_HORS-PROCESSUS,5478,10408,AUTRES,-,-,2024-01-01 00:00:00,2024-01-01 00:00:00,2050-12-31 00:00:00,2050-12-31 00:00:00,-,0,2300,2300,0,-
2,SI_HORS-PROCESSUS,5479,10409,AUTRES,-,-,2024-01-01 00:00:00,2024-01-01 00:00:00,2050-12-31 00:00:00,2050-12-31 00:00:00,-,0,2300,2300,0,-
3,SI_HORS-PROCESSUS,5476,10406,AUTRES,-,-,2024-01-01 00:00:00,2024-01-01 00:00:00,2050-12-31 00:00:00,2050-12-31 00:00:00,-,2300,2300,2300,0,-
4,INITIATIVE PODEV 2024,5535,10542,REALISATION,305,CDAD-BDX,2024-01-01 00:00:00,2024-01-01 00:00:00,2050-12-31 00:00:00,2024-09-30 00:00:00,-,0,2200,2200,2200,-


## Standardisation des colonnes texte

*Objectif : éviter les erreurs de casse, espaces, coquilles*.

In [2]:
cols_text = ["Centre", "NOM_SI", "Types_phases", "utilisateur_id2"]

for col in cols_text:
    df[col] = df[col].astype(str).str.strip().str.upper()


## Nettoyage des colonnes de charge

 *Problème détecté : certaines valeurs = "-" → doivent devenir NaN*

In [3]:
for col in ["ChargeTotaleEstimee", "ChargeTotaleActualisee"]:
    df[col] = df[col].replace("-", np.nan)      # remplacer '-'
    df[col] = df[col].replace("", np.nan)       # remplacer vide
    df[col] = df[col].astype(str).str.replace(",", ".", regex=False)  # si virgules
    df[col] = pd.to_numeric(df[col], errors="coerce")


C:\Users\Tenordem\AppData\Local\Temp\ipykernel_4848\82236366.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace("-", np.nan)      # remplacer '-'
C:\Users\Tenordem\AppData\Local\Temp\ipykernel_4848\82236366.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace("-", np.nan)      # remplacer '-'


## Détection des valeurs aberrantes

In [4]:
# Fonction IQR
def detect_outliers(s):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    return lower, upper

les_charges = ["ChargeTotaleEstimee", "ChargeTotaleActualisee"]

for col in les_charges:
    low, up = detect_outliers(df[col])
    print(col, " - bornes IQR :", low, up)
    print("Valeurs hors bornes :", (df[col] < low).sum() + (df[col] > up).sum())


ChargeTotaleEstimee  - bornes IQR : -45.0 75.0
Valeurs hors bornes : 1422
ChargeTotaleActualisee  - bornes IQR : -60.0 100.0
Valeurs hors bornes : 1379


Garder les outliers mais les surveiller.
Car parfois les projets très lourds sont vraiment lourds.

## Nettoyage des postes

In [5]:
df["utilisateur_id2"] = df["utilisateur_id2"].replace("-", "INCONNU")


## Nettoyage des centres et phases

In [6]:
print(df["Centre"].unique())
print(df["Types_phases"].unique())


['-' 'CDAD-BDX' 'CDAD-R' 'CDAD-T' 'PODEV-EM' 'CDAD-BO' 'DEMO1' 'FABNUM'
 'NAN' 'SOPRASTERIA']
['AUTRES' 'REALISATION' 'MAINTENANCE' 'ETUDE' '-']


In [7]:
# Nettoyage des centres
df["Centre"] = df["Centre"].astype(str).str.strip().str.upper()

df["Centre"] = df["Centre"].replace({
    "-": "INCONNU",
    "NAN": "INCONNU",
})

# Nettoyage des phases
df["Types_phases"] = df["Types_phases"].astype(str).str.strip().str.upper()

df["Types_phases"] = df["Types_phases"].replace({
    "-": "INCONNU"
})


In [8]:
print(df["Centre"].unique())
print(df["Types_phases"].unique())


['INCONNU' 'CDAD-BDX' 'CDAD-R' 'CDAD-T' 'PODEV-EM' 'CDAD-BO' 'DEMO1'
 'FABNUM' 'SOPRASTERIA']
['AUTRES' 'REALISATION' 'MAINTENANCE' 'ETUDE' 'INCONNU']


## NETTOYAGE DES CHARGES


In [9]:
for col in ["ChargeTotaleEstimee", "ChargeTotaleActualisee"]:
    df[col] = df[col].replace("-", np.nan)       # enlever '-'
    df[col] = df[col].replace("", np.nan)        # enlever vide
    df[col] = df[col].astype(str).str.replace(",", ".", regex=False)  # convertir 12,5 → 12.5
    df[col] = pd.to_numeric(df[col], errors="coerce")  # forcer en float


In [10]:
df[["ChargeTotaleEstimee", "ChargeTotaleActualisee"]].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7915 entries, 0 to 7914
Data columns (total 2 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ChargeTotaleEstimee     7853 non-null   float64
 1   ChargeTotaleActualisee  7894 non-null   float64
dtypes: float64(2)
memory usage: 123.8 KB


## NETTOYAGE DES POSTES

In [15]:
df["utilisateur_id2"] = df["utilisateur_id2"].replace("-", "INCONNU")


## CRÉATION DES INDICATEURS MÉTIER

Écart de charge

In [11]:
df["EcartCharge"] = df["ChargeTotaleActualisee"] - df["ChargeTotaleEstimee"]


Taux de surcharge

In [32]:
def calc_taux_surcharge(estimee, reelle):
    if estimee == 0 and reelle == 0:
        return 0
    if estimee == 0 and reelle > 0:
        return np.nan
    return reelle / estimee


In [33]:
df["TauxSurcharge"] = df.apply(lambda row: calc_taux_surcharge(row["ChargeTotaleEstimee"], row["ChargeTotaleActualisee"]), axis=1)
df["TauxSurcharge_%"] = df["TauxSurcharge"] * 100


In [27]:
df["TauxSurcharge"] = np.where(
    df["ChargeTotaleEstimee"] == 0,
    np.nan,
    df["ChargeTotaleActualisee"] / df["ChargeTotaleEstimee"]
)


Taux de surcharge (%)

In [28]:
df["TauxSurcharge_%"] = df["TauxSurcharge"] * 100


# CONTROLE FINAL

In [34]:
df.info()
df.isna().sum()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7915 entries, 0 to 7914
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   NOM_SI                   7915 non-null   object 
 1   ModuleId                 7915 non-null   object 
 2   PhaseId                  7915 non-null   object 
 3   Types_phases             7915 non-null   object 
 4   utilisateur_id2          7915 non-null   object 
 5   Centre                   7915 non-null   object 
 6   DateDebutPrev            7915 non-null   object 
 7   DateDebutReelle          7915 non-null   object 
 8   DateFinPrevisionnelle    7915 non-null   object 
 9   DateFinReelle            7915 non-null   object 
 10  DateFinActualisée        7915 non-null   object 
 11  charge_previsionnelle_n  7915 non-null   object 
 12  ChargeTotaleEstimee      7853 non-null   float64
 13  ChargeTotaleActualisee   7894 non-null   float64
 14  ModuleChargeInitiale    

,ChargeTotaleEstimee,ChargeTotaleActualisee,EcartCharge,TauxSurcharge,TauxSurcharge_%
count,7853.000000,7894.000000,7853.000000,7770.000000,7770.000000
mean,101.092576,142.075437,41.717114,0.745860,74.585991
std,305.913973,413.103468,198.831882,2.071378,207.137797
min,0.000000,0.000000,-338.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,30.000000,40.000000,0.000000,1.000000,100.000000
max,2300.000000,3600.000000,2700.000000,66.000000,6600.000000


### Analyse des colonnes nettoyées

Après nettoyage, les statistiques montrent :

- Les colonnes de charge sont maintenant correctement converties en nombres
  et ne contiennent plus de valeurs invalides (“-”, textes, vide).
- La médiane des charges estimées et réelles est de 0 h, ce qui confirme que
  beaucoup de lignes correspondent à de petites interventions ou à des valeurs non renseignées.
- Les valeurs maximales (jusqu’à 3600 h) sont réalistes pour des projets de longue durée.
- Le taux de surcharge moyen est de +77 %, ce qui montre une tendance générale à la sous-estimation.
- Les taux extrêmes (> 1000 %) correspondent aux lignes où la charge estimée était de 0
  → ce qui crée mathématiquement des surcharges très élevées.

Conclusion :  
**Les colonnes sont propres et fiables. Nous pouvons maintenant passer à la création des indicateurs et aux agrégations.**


# Agrégation par centre


In [37]:
agg_centre = (
    df.groupby("Centre", as_index=False)
      .agg(
          ChargeEstimee=("ChargeTotaleEstimee", "sum"),
          ChargeReelle=("ChargeTotaleActualisee", "sum"),
          NbPostes=("utilisateur_id2", "nunique")
      )
)

agg_centre["EcartCharge"] = agg_centre["ChargeReelle"] - agg_centre["ChargeEstimee"]

agg_centre["TauxSurcharge"] = np.where(
    (agg_centre["ChargeEstimee"] == 0) & (agg_centre["ChargeReelle"] == 0),
    0,
    np.where(
        agg_centre["ChargeEstimee"] == 0,
        np.nan,
        agg_centre["ChargeReelle"] / agg_centre["ChargeEstimee"]
    )
)

agg_centre["TauxSurcharge_%"] = agg_centre["TauxSurcharge"] * 100

agg_centre


,Centre,ChargeEstimee,ChargeReelle,NbPostes,EcartCharge,TauxSurcharge,TauxSurcharge_%
0,CDAD-BDX,183768.0,226556.149780,108,42788.149780,1.232838,123.283787
1,CDAD-BO,157035.0,277537.357422,75,120502.357422,1.767360,176.735987
2,CDAD-R,107928.0,190495.989990,105,82567.989990,1.765028,176.502844
3,CDAD-T,294997.0,375826.000000,100,80829.000000,1.273999,127.399940
4,DEMO1,2268.0,2806.000000,15,538.000000,1.237213,123.721340
5,FABNUM,0.0,0.000000,1,0.000000,0.000000,0.000000
6,INCONNU,35336.0,35283.000000,3,-53.000000,0.998500,99.850011
7,PODEV-EM,12548.0,13039.000000,32,491.000000,1.039130,103.912974
8,SOPRASTERIA,0.0,0.000000,1,0.000000,0.000000,0.000000


## Agrégation par phase


In [38]:
agg_phase = (
    df.groupby("Types_phases", as_index=False)
      .agg(
          ChargeEstimee=("ChargeTotaleEstimee", "sum"),
          ChargeReelle=("ChargeTotaleActualisee", "sum")
      )
)

agg_phase["EcartCharge"] = agg_phase["ChargeReelle"] - agg_phase["ChargeEstimee"]

agg_phase["TauxSurcharge"] = np.where(
    (agg_phase["ChargeEstimee"] == 0) & (agg_phase["ChargeReelle"] == 0),
    0,
    np.where(
        agg_phase["ChargeEstimee"] == 0,
        np.nan,
        agg_phase["ChargeReelle"] / agg_phase["ChargeEstimee"]
    )
)

agg_phase["TauxSurcharge_%"] = agg_phase["TauxSurcharge"] * 100


agg_phase


,Types_phases,ChargeEstimee,ChargeReelle,EcartCharge,TauxSurcharge,TauxSurcharge_%
0,AUTRES,10931.0,1.093100e+04,0.000000,1.000000,100.000000
1,ETUDE,12688.0,1.574900e+04,3061.000000,1.241252,124.125158
2,INCONNU,0.0,0.000000e+00,0.000000,0.000000,0.000000
3,MAINTENANCE,26933.0,3.625450e+04,9321.500000,1.346100,134.609958
4,REALISATION,743328.0,1.058609e+06,315280.997192,1.424148,142.414788


## Agrégation par module (NOM_SI)


In [39]:
agg_module = (
    df.groupby("NOM_SI", as_index=False)
      .agg(
          ChargeEstimee=("ChargeTotaleEstimee", "sum"),
          ChargeReelle=("ChargeTotaleActualisee", "sum"),
          NbPostes=("utilisateur_id2", "nunique")
      )
)

agg_module["EcartCharge"] = agg_module["ChargeReelle"] - agg_module["ChargeEstimee"]

agg_module["TauxSurcharge"] = np.where(
    (agg_module["ChargeEstimee"] == 0) & (agg_module["ChargeReelle"] == 0),
    0,
    np.where(
        agg_module["ChargeEstimee"] == 0,
        np.nan,
        agg_module["ChargeReelle"] / agg_module["ChargeEstimee"]
    )
)

agg_module["TauxSurcharge_%"] = agg_module["TauxSurcharge"] * 100


agg_module.head()


,NOM_SI,ChargeEstimee,ChargeReelle,NbPostes,EcartCharge,TauxSurcharge,TauxSurcharge_%
0,-,0.0,0.0,17,0.0,0.000000,0.000000
1,ACCES FORT KB,830.0,830.0,1,0.0,1.000000,100.000000
2,ACCT NG,6773.0,6773.0,12,0.0,1.000000,100.000000
3,ACHERON,288.0,348.0,9,60.0,1.208333,120.833333
4,ACTIVITES_DEFENSE,0.0,0.0,2,0.0,0.000000,0.000000


## Agrégation Centre × Phase


In [40]:
agg_centre_phase = (
    df.groupby(["Centre", "Types_phases"], as_index=False)
      .agg(
          ChargeEstimee=("ChargeTotaleEstimee", "sum"),
          ChargeReelle=("ChargeTotaleActualisee", "sum")
      )
)

agg_centre_phase["EcartCharge"] = agg_centre_phase["ChargeReelle"] - agg_centre_phase["ChargeEstimee"]

agg_centre_phase["TauxSurcharge"] = np.where(
    (agg_centre_phase["ChargeEstimee"] == 0) & (agg_centre_phase["ChargeReelle"] == 0),
    0,
    np.where(
        agg_centre_phase["ChargeEstimee"] == 0,
        np.nan,
        agg_centre_phase["ChargeReelle"] / agg_centre_phase["ChargeEstimee"]
    )
)

agg_centre_phase["TauxSurcharge_%"] = agg_centre_phase["TauxSurcharge"] * 100


agg_centre_phase.head()


,Centre,Types_phases,ChargeEstimee,ChargeReelle,EcartCharge,TauxSurcharge,TauxSurcharge_%
0,CDAD-BDX,AUTRES,40.0,40.00000,0.00000,1.000000,100.000000
1,CDAD-BDX,ETUDE,2320.0,3173.00000,853.00000,1.367672,136.767241
2,CDAD-BDX,MAINTENANCE,7559.0,8780.50000,1221.50000,1.161595,116.159545
3,CDAD-BDX,REALISATION,173849.0,214562.64978,40713.64978,1.234190,123.418973
4,CDAD-BO,AUTRES,0.0,0.00000,0.00000,0.000000,0.000000


## Agrégation Module × Phase


In [41]:
agg_module_phase = (
    df.groupby(["NOM_SI", "Types_phases"], as_index=False)
      .agg(
          ChargeEstimee=("ChargeTotaleEstimee", "sum"),
          ChargeReelle=("ChargeTotaleActualisee", "sum")
      )
)

agg_module_phase["EcartCharge"] = agg_module_phase["ChargeReelle"] - agg_module_phase["ChargeEstimee"]

agg_module_phase["TauxSurcharge"] = np.where(
    (agg_module_phase["ChargeEstimee"] == 0) & (agg_module_phase["ChargeReelle"] == 0),
    0,
    np.where(
        agg_module_phase["ChargeEstimee"] == 0,
        np.nan,
        agg_module_phase["ChargeReelle"] / agg_module_phase["ChargeEstimee"]
    )
)

agg_module_phase["TauxSurcharge_%"] = agg_module_phase["TauxSurcharge"] * 100


agg_module_phase.head()


,NOM_SI,Types_phases,ChargeEstimee,ChargeReelle,EcartCharge,TauxSurcharge,TauxSurcharge_%
0,-,INCONNU,0.0,0.0,0.0,0.0,0.0
1,ACCES FORT KB,ETUDE,30.0,30.0,0.0,1.0,100.0
2,ACCES FORT KB,MAINTENANCE,0.0,0.0,0.0,0.0,0.0
3,ACCES FORT KB,REALISATION,800.0,800.0,0.0,1.0,100.0
4,ACCT NG,ETUDE,0.0,0.0,0.0,0.0,0.0


In [42]:
with pd.ExcelWriter(r"C:\Users\Tenordem\Documents\projet-repartition-charge\data\processed\data_aggregations.xlsx") as writer:
    agg_centre.to_excel(writer, sheet_name="par_centre", index=False)
    agg_phase.to_excel(writer, sheet_name="par_phase", index=False)
    agg_module.to_excel(writer, sheet_name="par_module", index=False)
    agg_centre_phase.to_excel(writer, sheet_name="par_centre_phase", index=False)
    agg_module_phase.to_excel(writer, sheet_name="par_module_phase", index=False)
